# Electromagnetic waves (FDTD)

*em · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/em/em_wave_fdtd)

Maxwell's two curl equations marched forward in time on a Yee grid. The curl is written as $\varepsilon_{ijk}\,\partial_j F_k$: a table of finite differences contracted with the Levi-Civita symbol in one einsum. A wave from a point or line source hits a dielectric block, slows down, bends and partly reflects, while a damping border swallows what leaves the box.

**Vector form**

$$\partial_t\mathbf H = -\nabla\times\mathbf E,\qquad \varepsilon_r\,\partial_t\mathbf E = \nabla\times\mathbf H$$

**Index (tensor) form**

$$\partial_t H_i = -\varepsilon_{ijk}\,\partial_j E_k,\qquad \varepsilon_r\,\partial_t E_i = \varepsilon_{ijk}\,\partial_j H_k,\qquad c\,\Delta t \le \Delta x/\sqrt{2}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $E_i,\ H_i$ | `E, H` | (3, Nx, Ny) | field components on the grid; TM mode: only $E_z$, $H_x$, $H_y$ are non-zero |
| $\partial_j F_k$ | `forward_diff(F, h) / backward_diff(F, h)` | (3, 3, Nx, Ny) | table of finite differences; row $j = z$ is zero in 2D |
| $\varepsilon_{ijk}$ | `levi_civita(dev)` | (3, 3, 3) | Levi-Civita symbol |
| $\varepsilon_{ijk}\partial_j F_k$ | `einsum('ijk,jk...->i...', eps, D)` | (3, Nx, Ny) | repeated $j, k$ ⇒ summed: the curl at every cell |
| $\varepsilon_r(x, y)$ | `permittivity(Nx, Ny, eps_block, dev)` | (Nx, Ny) | relative permittivity: 1 in vacuum, $\varepsilon_\text{block}$ in the block ($n = \sqrt{\varepsilon_r}$) |
| $\Delta t$ | `courant_time_step(h, S, dim=2)` | scalar | time step $S\,\Delta x/(c\sqrt2)$; $S > 1$ is refused because the scheme would blow up |

Units with $c = \varepsilon_0 = \mu_0 = 1$ and cell size $\Delta x = 1$. $\mathbf E$ and $\mathbf H$ sit half a cell apart (forward differences for $\partial\mathbf E$, backward for $\partial\mathbf H$) and are updated alternately — the leap-frog Yee scheme. The source is "soft": it adds to $E_z$, so reflected waves pass straight through it. The absorbing border simply multiplies the fields by a factor slightly below 1 that grows smoothly towards the walls; real codes use a perfectly matched layer (PML) instead. Inside the block the wavelength shrinks by $n = \sqrt{\varepsilon_r}$; at normal incidence a fraction $(1 - n)/(1 + n)$ of the amplitude reflects.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/em/em_wave_fdtd/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors
from .equation import courant_time_step

NX, NY, BORDER = 200, 130, 20  # grid cells and width of the absorbing border; cell size h = 1


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    eps = tensors.levi_civita(dev)  # (3, 3, 3)
    eps_r = tensors.permittivity(NX, NY, params["eps_block"], dev)  # (Nx, Ny)
    damping = tensors.damping(NX, NY, BORDER, 0.1, dev)  # (Nx, Ny)
    E, H = tensors.empty_fields(NX, NY, dev)  # (3, Nx, Ny) each
    mask = solve.point_source(NX, NY, dev) if params["source"] == "point" else solve.line_source(NX, NY, dev)

    dt = courant_time_step(1.0, params["courant"], dim=2)
    n_steps = int(params["t_max"] / dt)
    waveform = solve.continuous_wave(params["wavelength"])
    times, Ez = solve.simulate(E, H, eps_r, damping, mask, waveform, eps, dt, 1.0, n_steps, save_every=4)

    n_block = params["eps_block"] ** 0.5
    print(f"grid {NX}x{NY}, dt = {dt:.3f} (Courant number {params['courant']}), {n_steps} steps")
    reflection = (1 - n_block) / (1 + n_block)  # amplitude reflected at normal incidence
    print(
        f"block: n = {n_block:.2f}, wavelength {params['wavelength']:.0f} -> {params['wavelength'] / n_block:.1f} cells"
    )
    print(f"normal-incidence reflection coefficient r = (1 - n)/(1 + n) = {reflection:+.2f}")
    return plot.animate(times, Ez, eps_r, BORDER, out_dir / "em_wave_fdtd.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/em/em_wave_fdtd/equation.py
r"""Physics → tensor translation.

Maxwell's curl equations in a medium with relative permittivity ε_r (units with c = ε0 = μ0 = 1):

    \partial_t \mathbf H = -\nabla \times \mathbf E,     \varepsilon_r \partial_t \mathbf E = \nabla \times \mathbf H

The curl is the Levi-Civita symbol contracted with the derivative and the field:

    \partial_t H_i = -\varepsilon_{ijk} \partial_j E_k
    \varepsilon_r \partial_t E_i = \varepsilon_{ijk} \partial_j H_k

j and k are repeated, so both are summed. We store the derivative table D_{jk} = ∂_j F_k as a tensor of
shape (3, 3, Nx, Ny) and contract it with ε_{ijk}: einsum('ijk,jk...->i...').

The problem is 2D (nothing depends on z, so ∂_z = 0). With E = (0, 0, E_z) and H = (H_x, H_y, 0) — the TM mode —
the curls only ever produce those same components, so the TM mode stays TM.

Yee scheme: E and H live half a cell apart in space and half a step apart in time. ∂_j E_k uses forward
differences (E_{n+1} − E_n), ∂_j H_k backward differences (H_n − H_{n−1}), which makes every derivative a
centred one. The update is explicit and only stable under the Courant condition

    c \, \Delta t \le \Delta x / \sqrt{d}        (d = number of space dimensions)
"""

import math

import torch


def forward_diff(F: torch.Tensor, h: float) -> torch.Tensor:
    """D_{jk} = ∂_j F_k by forward differences.   F: (3, Nx, Ny) → (3, 3, Nx, Ny); row j = z stays 0 (∂_z = 0)."""
    D = torch.zeros(3, *F.shape, dtype=F.dtype, device=F.device)
    D[0, :, :-1, :] = (F[:, 1:, :] - F[:, :-1, :]) / h  # ∂_x
    D[1, :, :, :-1] = (F[:, :, 1:] - F[:, :, :-1]) / h  # ∂_y
    return D


def backward_diff(F: torch.Tensor, h: float) -> torch.Tensor:
    """D_{jk} = ∂_j F_k by backward differences.   (3, Nx, Ny) → (3, 3, Nx, Ny)"""
    D = torch.zeros(3, *F.shape, dtype=F.dtype, device=F.device)
    D[0, :, 1:, :] = (F[:, 1:, :] - F[:, :-1, :]) / h
    D[1, :, :, 1:] = (F[:, :, 1:] - F[:, :, :-1]) / h
    return D


def curl(eps: torch.Tensor, D: torch.Tensor) -> torch.Tensor:
    r"""(∇×F)_i = ε_{ijk} ∂_j F_k   — j and k summed.   eps: (3, 3, 3), D: (3, 3, Nx, Ny) → (3, Nx, Ny)"""
    return torch.einsum("ijk,jk...->i...", eps, D)


def courant_time_step(h: float, courant: float, dim: int, c: float = 1.0) -> float:
    """Δt = S · Δx / (c √d) for a Courant number S. S > 1 would blow up, so it is refused."""
    if not 0 < courant <= 1:
        raise ValueError(f"Courant number must be in (0, 1], got {courant}: the scheme would be unstable")
    return courant * h / (c * math.sqrt(dim))


def yee_step(E, H, eps, eps_r, damping, dt: float, h: float) -> tuple[torch.Tensor, torch.Tensor]:
    """One leap-frog step: H from curl E, then E from the new curl H.

    E, H: (3, Nx, Ny);  eps_r, damping: (Nx, Ny) — damping is 1 inside and drops below 1 in the absorbing border.
    """
    H = damping * (H - dt * curl(eps, forward_diff(E, h)))  # ∂_t H_i = −ε_ijk ∂_j E_k
    E = damping * (E + dt / eps_r * curl(eps, backward_diff(H, h)))  # ε_r ∂_t E_i = ε_ijk ∂_j H_k
    return E, H


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/em/em_wave_fdtd/tensors.py
"""Build the concrete tensors: Levi-Civita symbol, permittivity map, absorbing border and empty fields."""

import itertools

import torch

EVEN = [(0, 1, 2), (1, 2, 0), (2, 0, 1)]  # cyclic permutations of (x, y, z)


def levi_civita(device: torch.device) -> torch.Tensor:
    """ε_ijk of shape (3, 3, 3): +1 for even permutations, −1 for odd ones, 0 when an index repeats."""
    eps = torch.zeros(3, 3, 3, device=device)
    for i, j, k in itertools.permutations(range(3)):
        eps[i, j, k] = 1.0 if (i, j, k) in EVEN else -1.0
    return eps


def permittivity(nx: int, ny: int, eps_block: float, device: torch.device) -> torch.Tensor:
    """ε_r(x, y) of shape (Nx, Ny): vacuum (1) with a dielectric block in the right half."""
    eps_r = torch.ones(nx, ny, device=device)
    eps_r[int(0.55 * nx) : int(0.8 * nx), int(0.25 * ny) : int(0.85 * ny)] = eps_block
    return eps_r


def damping(nx: int, ny: int, width: int, strength: float, device: torch.device) -> torch.Tensor:
    """Absorbing border, shape (Nx, Ny): 1 inside, falling smoothly to 1 − strength at the walls.

    Multiplying the fields by it every step soaks up outgoing waves. The profile rises quadratically, so the
    wave meets no sharp edge to reflect from. Set ny = 1 for a 1D run: then only the x walls absorb.
    """

    def profile(n: int) -> torch.Tensor:
        depth = torch.arange(n, device=device, dtype=torch.float32)
        distance_in = torch.minimum(depth, n - 1 - depth)  # cells from the nearest wall
        inside = ((width - distance_in) / width).clamp(min=0.0)  # 0 inside, → 1 at the wall
        return 1.0 - strength * inside**2

    if ny == 1:
        return profile(nx)[:, None]
    return profile(nx)[:, None] * profile(ny)[None, :]


def empty_fields(nx: int, ny: int, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """E_i and H_i, each of shape (3, Nx, Ny), all zero. TM mode: only E_z, H_x and H_y will become non-zero."""
    return torch.zeros(3, nx, ny, device=device), torch.zeros(3, nx, ny, device=device)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/em/em_wave_fdtd/solve.py
"""March the Yee scheme in time with a soft source, keeping snapshots of E_z for the animation."""

import math
from typing import Callable

import torch

from .equation import yee_step


def continuous_wave(wavelength: float, c: float = 1.0) -> Callable[[float], float]:
    """sin(ω t) with ω = 2πc/λ, switched on smoothly over about two periods (a sudden start sprays high frequencies)."""
    omega = 2 * math.pi * c / wavelength
    period = 2 * math.pi / omega
    return lambda t: math.sin(omega * t) * (1 - math.exp(-((t / (2 * period)) ** 2)))


def gaussian_pulse(t0: float, width: float) -> Callable[[float], float]:
    """A single bump exp(−((t − t0)/width)²), used by the 1D checks."""
    return lambda t: math.exp(-(((t - t0) / width) ** 2))


def simulate(E, H, eps_r, damping, source_mask, waveform, eps, dt: float, h: float, n_steps: int, save_every=1):
    """Start from fields E, H (3, Nx, Ny); return (times (F,), E_z snapshots (F, Nx, Ny)).

    source_mask (Nx, Ny) says where the source adds to E_z ("soft" source: the wave passes straight through it).
    """
    times, frames = [], []
    for n in range(n_steps):
        E, H = yee_step(E, H, eps, eps_r, damping, dt, h)
        E[2] += source_mask * waveform((n + 1) * dt)
        if n % save_every == 0:
            times.append((n + 1) * dt)
            frames.append(E[2].clone())
    return torch.tensor(times), torch.stack(frames)


def point_source(nx: int, ny: int, device) -> torch.Tensor:
    """Source mask (Nx, Ny): a single cell a quarter of the way in, slightly below the middle."""
    mask = torch.zeros(nx, ny, device=device)
    mask[nx // 4, int(0.45 * ny)] = 1.0
    return mask


def line_source(nx: int, ny: int, device) -> torch.Tensor:
    """Source mask (Nx, Ny): a whole column of cells, which launches a plane wave travelling along ±x."""
    mask = torch.zeros(nx, ny, device=device)
    mask[nx // 4, :] = 0.3
    return mask


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/em/em_wave_fdtd/plot.py
"""Animate E_z as a heatmap, with the dielectric block and the absorbing border outlined."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.patches import Rectangle

from problems._common.plotting import frame_indices, save_gif


def animate(times: torch.Tensor, Ez: torch.Tensor, eps_r: torch.Tensor, border: int, path: Path) -> Path:
    t = times.numpy()  # (F,)
    field = Ez.cpu().numpy()  # (F, Nx, Ny)
    nx, ny = field.shape[1:]
    limit = (
        np.quantile(np.abs(field[-1]), 0.99) + 1e-9
    )  # colour scale from the final wave (16 colours keep the GIF small)

    fig, ax = plt.subplots(figsize=(7, 4))
    image = ax.imshow(
        field[0].T, origin="lower", cmap=plt.get_cmap("RdBu_r", 16), vmin=-limit, vmax=limit, extent=(0, nx, 0, ny),
        interpolation="nearest",
    )  # fmt: skip
    # the dielectric block: wherever ε_r > 1
    rows, cols = np.nonzero(eps_r.cpu().numpy() > 1)
    if len(rows):
        x0, y0 = rows.min(), cols.min()
        ax.add_patch(Rectangle((x0, y0), rows.max() + 1 - x0, cols.max() + 1 - y0, fill=False, ec="k", lw=1.2))
        ax.text(x0 + 2, cols.max() - 2, f"ε_r = {eps_r.max():g}", va="top", fontsize=9)
    ax.add_patch(
        Rectangle((border, border), nx - 2 * border, ny - 2 * border, fill=False, ec="#9ca3af", ls="--", lw=0.8)
    )
    ax.set_xlabel("x (cells)")
    ax.set_ylabel("y (cells)")
    ax.grid(False)
    fig.colorbar(image, ax=ax, label="E_z", shrink=0.8)
    title = ax.set_title("")

    def update(step: int):
        image.set_data(field[step].T)  # transpose: x runs along the horizontal axis
        title.set_text(f"TM wave E_z  (dashed: absorbing border)   t = {t[step]:.0f}")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t), 75), path, fps=15)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'source': 'point',
    'eps_block': 4.0,
    'wavelength': 14.0,
    'courant': 0.7,
    't_max': 220.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.em.em_wave_fdtd")]:
    del sys.modules[name]
main = importlib.import_module("problems.em.em_wave_fdtd.main")
problem_dir = Path("problems/em/em_wave_fdtd")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
